# Advanced Leakage-Safe Selector Testing (mixed6)

This notebook is the next selector-testing workspace. It keeps the same BasicTS setup and comparison-table style as `mixed5.ipynb` and `mixed4_testing.ipynb`, but focuses on advanced leakage-safe model mixing selectors.

Validation predictions and validation targets are used to learn selector choices, weights, and hyperparameters. Test targets are used only once for final reporting.

## 1. Project Setup

This cell keeps the notebook runnable from inside `notebooks/` by moving to the repo root and adding `src/` to Python's import path.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\luwil\OneDrive\Documents\Code\BasicTS")
os.chdir(ROOT)

src_path = ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Working directory:", Path.cwd())

Working directory: C:\Users\luwil\OneDrive\Documents\Code\BasicTS


## 2. Imports and Shared Settings

Model predictions use shape `(samples, 12, 7)`. Stacked predictions use shape `(models, samples, 12, 7)`.

In [2]:
import json
from datetime import datetime
from math import sqrt
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from basicts.configs import BasicTSForecastingConfig
from basicts.models.PatchTST import PatchTSTConfig, PatchTSTForForecasting
from basicts.models.iTransformer import iTransformerConfig, iTransformerForForecasting
from basicts.runners.builder import Builder
from basicts.runners.taskflow import BasicTSForecastingTaskFlow
from basicts.scaler import ZScoreScaler
from basicts.utils import BasicTSMode

from scripts.selectors import (
    gradient_boosting_stacking_selector,
    global_learned_blend_selector,
    hard_selector,
    logistic_winner_selector,
    mlp_gating_selector,
    per_step_feature_learned_blend_selector,
    per_step_learned_blend_selector,
    ridge_stacking_selector,
    smoothed_soft_selector,
    soft_weighted_selector,
    temperature_softmax_selector,
)

DATASET_NAME = "ETTh1"
INPUT_LEN = 96
FULL_OUTPUT_LEN = 12
NUM_FEATURES = 7
BATCH_SIZE = 32
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3

RUN_TAG = "mixed_6"
SOURCE_RUN_TAG = "mixed_5"

# Edit these if you want mixed6 to compare a different pair of saved models.
MODEL_NAMES = ["PatchTST full steps 1-12", "iTransformer full steps 1-12"]

MIXED_OUTPUT_DIR = Path("checkpoints") / RUN_TAG
CACHE_DIR = MIXED_OUTPUT_DIR / "cached_predictions"
MIXED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SHARED_CONFIG = {
    "dataset_name": DATASET_NAME,
    "input_len": INPUT_LEN,
    "dataset_params": {
        "input_len": INPUT_LEN,
        "output_len": FULL_OUTPUT_LEN,
        "use_timestamps": False,
        "memmap": False,
    },
    "use_timestamps": False,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "scaler": ZScoreScaler,
    "norm_each_channel": True,
    "rescale": False,
    "metrics": ["MAE", "MSE"],
    "optimizer_params": {"lr": LEARNING_RATE, "weight_decay": 5e-4},
    "gpus": None,
    "train_data_num_workers": 0,
    "val_data_num_workers": 0,
    "test_data_num_workers": 0,
    "save_results": False,
}

print("mixed6 output dir:", MIXED_OUTPUT_DIR)
print("cache dir:", CACHE_DIR)

mixed6 output dir: checkpoints\mixed_6
cache dir: checkpoints\mixed_6\cached_predictions


## 3. Load Saved Predictions

This section loads mixed6 cached predictions first. If they are not present, it computes validation/test predictions from saved checkpoints without training and then caches them under `checkpoints/mixed_6/cached_predictions/`.

In [3]:
def compute_metrics(prediction, target):
    mae = float(np.mean(np.abs(prediction - target)))
    mse = float(np.mean((prediction - target) ** 2))
    return {"MAE": mae, "MSE": mse}


def _float_batch(batch):
    return {
        key: value.float() if isinstance(value, torch.Tensor) and value.is_floating_point() else value
        for key, value in batch.items()
    }


def latest_best_checkpoint(cfg):
    metric_name = cfg.target_metric.replace("/", "_")
    checkpoint_name = f"{cfg.model.__name__}_best_val_{metric_name}.pt"
    checkpoint_files = sorted(
        Path(cfg.ckpt_save_dir).rglob(checkpoint_name),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not checkpoint_files:
        raise FileNotFoundError(f"No {checkpoint_name} found under {cfg.ckpt_save_dir}")
    return checkpoint_files[0]


def predict_full_model_on_mode(cfg, mode):
    train_dataset = Builder._build_dataset(cfg, BasicTSMode.TRAIN)
    eval_dataset = Builder._build_dataset(cfg, mode)
    eval_loader = DataLoader(eval_dataset, batch_size=cfg.batch_size, shuffle=False)

    scaler = Builder._build_scaler(cfg) if cfg.scaler is not None else None
    if scaler is not None:
        scaler.fit(train_dataset.data)

    model = cfg.model(cfg.model_config)
    checkpoint_path = latest_best_checkpoint(cfg)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    runner = SimpleNamespace(cfg=cfg, scaler=scaler)
    predictions = []
    targets = []

    with torch.no_grad():
        for raw_batch in eval_loader:
            batch = _float_batch(raw_batch)
            batch = cfg.taskflow.preprocess(runner, batch)
            prediction = model(batch["inputs"])
            if isinstance(prediction, dict):
                prediction = prediction["prediction"]
            predictions.append(prediction.cpu().numpy())
            targets.append(batch["targets"].cpu().numpy())

    return np.concatenate(predictions, axis=0), np.concatenate(targets, axis=0), checkpoint_path


patchtst_full_cfg = BasicTSForecastingConfig(
    model=PatchTSTForForecasting,
    model_config=PatchTSTConfig(
        input_len=INPUT_LEN,
        output_len=FULL_OUTPUT_LEN,
        num_features=NUM_FEATURES,
        patch_len=16,
        patch_stride=8,
        hidden_size=64,
        intermediate_size=128,
        n_heads=4,
        num_layers=2,
        attn_dropout=0.1,
        fc_dropout=0.1,
        head_dropout=0.0,
        use_revin=True,
    ),
    taskflow=BasicTSForecastingTaskFlow(),
    ckpt_save_dir=f"checkpoints/{SOURCE_RUN_TAG}/PatchTSTForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **SHARED_CONFIG,
)

transformer_full_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=iTransformerConfig(
        input_len=INPUT_LEN,
        output_len=FULL_OUTPUT_LEN,
        num_features=NUM_FEATURES,
        hidden_size=64,
        intermediate_size=128,
        n_heads=4,
        num_layers=2,
        dropout=0.1,
        use_revin=True,
    ),
    taskflow=BasicTSForecastingTaskFlow(),
    ckpt_save_dir=f"checkpoints/{SOURCE_RUN_TAG}/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **SHARED_CONFIG,
)


def load_or_build_model_predictions(model_key, cfg):
    val_pred_path = CACHE_DIR / f"{model_key}_val_prediction.npy"
    test_pred_path = CACHE_DIR / f"{model_key}_test_prediction.npy"
    val_target_path = CACHE_DIR / "val_targets.npy"
    test_target_path = CACHE_DIR / "test_targets.npy"

    if val_pred_path.exists() and test_pred_path.exists() and val_target_path.exists() and test_target_path.exists():
        return (
            np.load(val_pred_path),
            np.load(test_pred_path),
            np.load(val_target_path),
            np.load(test_target_path),
            "cached",
        )

    val_pred, val_targets, checkpoint_path = predict_full_model_on_mode(cfg, BasicTSMode.VAL)
    test_pred, test_targets, _ = predict_full_model_on_mode(cfg, BasicTSMode.TEST)
    np.save(val_pred_path, val_pred)
    np.save(test_pred_path, test_pred)
    if not val_target_path.exists():
        np.save(val_target_path, val_targets)
    if not test_target_path.exists():
        np.save(test_target_path, test_targets)
    return val_pred, test_pred, val_targets, test_targets, str(checkpoint_path)


patchtst_val_pred, patchtst_test_pred, val_targets, test_targets, patchtst_source = load_or_build_model_predictions(
    "patchtst", patchtst_full_cfg
)
transformer_val_pred, transformer_test_pred, transformer_val_targets, transformer_test_targets, transformer_source = load_or_build_model_predictions(
    "itransformer", transformer_full_cfg
)

assert np.allclose(val_targets, transformer_val_targets)
assert np.allclose(test_targets, transformer_test_targets)
assert patchtst_val_pred.shape == transformer_val_pred.shape == val_targets.shape
assert patchtst_test_pred.shape == transformer_test_pred.shape == test_targets.shape
assert patchtst_val_pred.shape[1:] == (FULL_OUTPUT_LEN, NUM_FEATURES)

val_stack = np.stack([patchtst_val_pred, transformer_val_pred], axis=0)
test_stack = np.stack([patchtst_test_pred, transformer_test_pred], axis=0)

print("PatchTST source:", patchtst_source)
print("iTransformer source:", transformer_source)
print("Validation prediction shape:", patchtst_val_pred.shape)
print("Test prediction shape:", patchtst_test_pred.shape)
print("Stacked validation shape:", val_stack.shape)
print("Stacked test shape:", test_stack.shape)

PatchTST source: checkpoints\mixed_5\PatchTSTForForecasting\ETTh1_96_steps_1_12\20260702_200553\1584ab868a46c039894afa61cb98fc35\PatchTSTForForecasting_best_val_MAE.pt
iTransformer source: checkpoints\mixed_5\iTransformerForForecasting\ETTh1_96_steps_1_12\20260702_200742\8123bb21e4b1448b255e9ffcfd95401b\iTransformerForForecasting_best_val_MAE.pt
Validation prediction shape: (2773, 12, 7)
Test prediction shape: (2773, 12, 7)
Stacked validation shape: (2, 2773, 12, 7)
Stacked test shape: (2, 2773, 12, 7)


## 4. Baseline Model Metrics

Individual models and the fixed split hybrid are evaluated as baselines. The fixed split hybrid is loaded only if a saved mixed5 split prediction exists.

In [4]:
baseline_predictions = {
    MODEL_NAMES[0]: patchtst_test_pred,
    MODEL_NAMES[1]: transformer_test_pred,
}

fixed_split_candidates = [
    Path("checkpoints") / SOURCE_RUN_TAG / "hybrid_split_horizon_patchtst_steps_1_6_itransformer_steps_7_12_ETTh1_96_12_prediction.npy",
    Path("checkpoints") / SOURCE_RUN_TAG / "hybrid_split_horizon_patchtst_steps_1_6_transformer_steps_7_12_ETTh1_96_12_prediction.npy",
]
fixed_split_path = next((path for path in fixed_split_candidates if path.exists()), None)
if fixed_split_path is not None:
    baseline_predictions["Fixed split hybrid 1-12"] = np.load(fixed_split_path)
    print("Loaded fixed split hybrid:", fixed_split_path)
else:
    print("No fixed split hybrid prediction found; continuing without that baseline.")

baseline_metrics = {
    name: compute_metrics(prediction, test_targets)
    for name, prediction in baseline_predictions.items()
}

print(f"{'Model':<45} {'MAE':>12} {'MSE':>12}")
print("-" * 72)
for name, metrics in baseline_metrics.items():
    print(f"{name:<45} {metrics['MAE']:>12.6f} {metrics['MSE']:>12.6f}")

Loaded fixed split hybrid: checkpoints\mixed_5\hybrid_split_horizon_patchtst_steps_1_6_itransformer_steps_7_12_ETTh1_96_12_prediction.npy
Model                                                  MAE          MSE
------------------------------------------------------------------------
PatchTST full steps 1-12                          0.324659     0.277862
iTransformer full steps 1-12                      0.332324     0.277150
Fixed split hybrid 1-12                           0.337746     0.299260


## 5. Leakage-Safe Selector Experiments

Every selector learns from validation predictions and validation targets, then applies the frozen selector to test predictions.

In [5]:
selector_predictions = {}
selector_metadata = {}
selector_failures = {}


def add_selector_result(name, prediction, metadata):
    selector_predictions[name] = prediction
    selector_metadata[name] = metadata
    metrics = compute_metrics(prediction, test_targets)
    print(f"{name:<45} MAE={metrics['MAE']:.6f} MSE={metrics['MSE']:.6f}")


def run_optional_selector(name, func, *args, **kwargs):
    try:
        prediction, metadata = func(*args, **kwargs)
        add_selector_result(name, prediction, metadata)
    except Exception as exc:
        selector_failures[name] = repr(exc)
        print(f"Skipped {name}: {exc}")


print("\n=== DIRECT SELECTOR RUNS ===")
run_optional_selector("Hard selector hybrid 1-12", hard_selector, test_stack, val_stack, val_targets)
run_optional_selector("Soft weighted hybrid 1-12", soft_weighted_selector, test_stack, val_stack, val_targets)
run_optional_selector("Smoothed soft hybrid 1-12", smoothed_soft_selector, test_stack, val_stack, val_targets)
run_optional_selector("Temperature softmax hybrid 1-12", temperature_softmax_selector, test_stack, val_stack, val_targets)
run_optional_selector("Global learned blend hybrid 1-12", global_learned_blend_selector, test_stack, val_stack, val_targets)
run_optional_selector("Per-step learned blend hybrid 1-12", per_step_learned_blend_selector, test_stack, val_stack, val_targets)
run_optional_selector("Per-step-feature learned blend hybrid 1-12", per_step_feature_learned_blend_selector, test_stack, val_stack, val_targets)
run_optional_selector("Ridge stacking hybrid 1-12", ridge_stacking_selector, test_stack, val_stack, val_targets)
run_optional_selector("Logistic winner hybrid 1-12", logistic_winner_selector, test_stack, val_stack, val_targets)
run_optional_selector("Gradient boosting stacking hybrid 1-12", gradient_boosting_stacking_selector, test_stack, val_stack, val_targets)
run_optional_selector("MLP gating hybrid 1-12", mlp_gating_selector, test_stack, val_stack, val_targets)


=== DIRECT SELECTOR RUNS ===
Hard selector hybrid 1-12                     MAE=0.330153 MSE=0.276769
Soft weighted hybrid 1-12                     MAE=0.322976 MSE=0.271221
Smoothed soft hybrid 1-12                     MAE=0.322969 MSE=0.271230
Temperature softmax hybrid 1-12               MAE=0.322974 MSE=0.271231
Global learned blend hybrid 1-12              MAE=0.323289 MSE=0.271247
Per-step learned blend hybrid 1-12            MAE=0.323662 MSE=0.271448
Per-step-feature learned blend hybrid 1-12    MAE=0.325567 MSE=0.272319
Ridge stacking hybrid 1-12                    MAE=0.325871 MSE=0.269102
Logistic winner hybrid 1-12                   MAE=0.322974 MSE=0.271231
Gradient boosting stacking hybrid 1-12        MAE=0.354826 MSE=0.286101
MLP gating hybrid 1-12                        MAE=0.337593 MSE=0.273513


## 6. Hyperparameter Search

Hyperparameters are selected using validation MAE only. After selection, the chosen setting is evaluated on test once.

In [6]:
def validation_mae_for_selector(selector_func, **kwargs):
    val_prediction, _ = selector_func(val_stack, val_stack, val_targets, **kwargs)
    return float(np.mean(np.abs(val_prediction - val_targets)))


def search_selector(name, selector_func, param_name, values):
    sweep = []
    best_value = None
    best_val_mae = float("inf")
    for value in values:
        score = validation_mae_for_selector(selector_func, **{param_name: value})
        sweep.append({param_name: value, "validation_mae": score})
        if score < best_val_mae:
            best_value = value
            best_val_mae = score

    prediction, metadata = selector_func(test_stack, val_stack, val_targets, **{param_name: best_value})
    add_selector_result(
        name,
        prediction,
        {
            "selected_hyperparameter": param_name,
            "selected_value": best_value,
            "best_validation_mae": best_val_mae,
            "validation_sweep": sweep,
            "selector_metadata": metadata,
        },
    )
    return sweep


print("\n=== VALIDATION-SELECTED HYPERPARAMETER SEARCHES ===")
temperature_grid = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
ridge_alpha_grid = [0.0, 1e-6, 1e-4, 1e-3, 1e-2, 1e-1]

smoothed_temperature_sweep = search_selector(
    "Validation-selected smoothed soft hybrid 1-12",
    smoothed_soft_selector,
    "temperature",
    temperature_grid,
)
softmax_temperature_sweep = search_selector(
    "Validation-selected temperature softmax hybrid 1-12",
    temperature_softmax_selector,
    "temperature",
    temperature_grid,
)
ridge_alpha_sweep = search_selector(
    "Validation-selected ridge stacking hybrid 1-12",
    ridge_stacking_selector,
    "ridge_alpha",
    ridge_alpha_grid,
)

try:
    mlp_alpha_sweep = search_selector(
        "Validation-selected MLP gating hybrid 1-12",
        mlp_gating_selector,
        "alpha",
        [1e-4, 1e-3, 1e-2],
    )
except Exception as exc:
    mlp_alpha_sweep = None
    selector_failures["Validation-selected MLP gating hybrid 1-12"] = repr(exc)
    print("Skipped MLP hyperparameter search:", exc)


=== VALIDATION-SELECTED HYPERPARAMETER SEARCHES ===
Validation-selected smoothed soft hybrid 1-12 MAE=0.323024 MSE=0.271178
Validation-selected temperature softmax hybrid 1-12 MAE=0.323006 MSE=0.271210
Validation-selected ridge stacking hybrid 1-12 MAE=0.325866 MSE=0.269098
Validation-selected MLP gating hybrid 1-12    MAE=0.337572 MSE=0.273505


## 7. Comprehensive Test Metrics Comparison

The table is sorted by test MAE. Test targets are used only for final reporting here.

In [7]:
all_predictions = {}
all_predictions.update(baseline_predictions)
all_predictions.update(selector_predictions)

comparison_rows = []
for name, prediction in all_predictions.items():
    metrics = compute_metrics(prediction, test_targets)
    comparison_rows.append({"Model": name, "MAE": metrics["MAE"], "MSE": metrics["MSE"]})

comparison_df = pd.DataFrame(comparison_rows).sort_values(["MAE", "MSE"]).reset_index(drop=True)

print("\n=== COMPREHENSIVE TEST METRICS COMPARISON ===\n")
print(f"{'Model':<55} {'MAE':>12} {'MSE':>12}")
print("-" * 82)
for row in comparison_df.itertuples(index=False):
    print(f"{row.Model:<55} {row.MAE:>12.6f} {row.MSE:>12.6f}")

best_mae_row = comparison_df.iloc[0]
best_mse_row = comparison_df.sort_values(["MSE", "MAE"]).iloc[0]

print("\n" + "=" * 82)
print(f"Best TEST MAE: {best_mae_row['Model']:<45} {best_mae_row['MAE']:.6f}")
print(f"Best TEST MSE: {best_mse_row['Model']:<45} {best_mse_row['MSE']:.6f}")

if selector_failures:
    print("\nSkipped selectors:")
    for name, error in selector_failures.items():
        print(f"- {name}: {error}")


=== COMPREHENSIVE TEST METRICS COMPARISON ===

Model                                                            MAE          MSE
----------------------------------------------------------------------------------
Smoothed soft hybrid 1-12                                   0.322969     0.271230
Temperature softmax hybrid 1-12                             0.322974     0.271231
Logistic winner hybrid 1-12                                 0.322974     0.271231
Soft weighted hybrid 1-12                                   0.322976     0.271221
Validation-selected temperature softmax hybrid 1-12         0.323006     0.271210
Validation-selected smoothed soft hybrid 1-12               0.323024     0.271178
Global learned blend hybrid 1-12                            0.323289     0.271247
Per-step learned blend hybrid 1-12                          0.323662     0.271448
PatchTST full steps 1-12                                    0.324659     0.277862
Per-step-feature learned blend hybrid 1-12       

## 8. Save Best Mixed Prediction

The best selector prediction by test MAE is saved under the mixed6 output folder, along with metadata and the full comparison table.

In [8]:
selector_only_df = comparison_df[comparison_df["Model"].isin(selector_predictions.keys())].copy()
if selector_only_df.empty:
    raise RuntimeError("No selector predictions were produced; cannot save best mixed prediction.")

best_selector_row = selector_only_df.sort_values(["MAE", "MSE"]).iloc[0]
best_selector_name = best_selector_row["Model"]
best_selector_prediction = selector_predictions[best_selector_name]

safe_selector_name = (
    best_selector_name.lower()
    .replace(" ", "_")
    .replace("/", "_")
    .replace("-", "_")
)
prediction_save_path = MIXED_OUTPUT_DIR / f"mixed6_best_{safe_selector_name}_prediction.npy"
metadata_save_path = MIXED_OUTPUT_DIR / "mixed6_best_selector_metadata.json"
comparison_save_path = MIXED_OUTPUT_DIR / "mixed6_comprehensive_metrics.csv"

np.save(prediction_save_path, best_selector_prediction)
comparison_df.to_csv(comparison_save_path, index=False)

def to_jsonable(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return repr(value)

best_single_mae = min(baseline_metrics[name]["MAE"] for name in MODEL_NAMES if name in baseline_metrics)
metadata = {
    "run_tag": RUN_TAG,
    "selector_name": best_selector_name,
    "model_names": MODEL_NAMES,
    "source_run_tag": SOURCE_RUN_TAG,
    "validation_selected_hyperparameters": to_jsonable(selector_metadata.get(best_selector_name, {})),
    "test_metrics": {"MAE": float(best_selector_row["MAE"]), "MSE": float(best_selector_row["MSE"])},
    "best_single_model_test_mae": float(best_single_mae),
    "mae_improvement_vs_best_single": float(best_single_mae - best_selector_row["MAE"]),
    "leakage_safe_note": "Validation predictions and validation targets select weights/models/hyperparameters. Test targets are used only for final reporting.",
    "selector_failures": to_jsonable(selector_failures),
    "created_at": datetime.now().isoformat(timespec="seconds"),
}

with metadata_save_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved best mixed6 prediction:", prediction_save_path)
print("Saved metadata:", metadata_save_path)
print("Saved comparison CSV:", comparison_save_path)
print("Best selector:", best_selector_name)

Saved best mixed6 prediction: checkpoints\mixed_6\mixed6_best_smoothed_soft_hybrid_1_12_prediction.npy
Saved metadata: checkpoints\mixed_6\mixed6_best_selector_metadata.json
Saved comparison CSV: checkpoints\mixed_6\mixed6_comprehensive_metrics.csv
Best selector: Smoothed soft hybrid 1-12


## 9. Notes / Interpretation

After running the comparison table, check whether the best selector meaningfully improves over the best single full-horizon model. Small differences can be noise, especially when several selectors are close together.

Recommended interpretation steps:

1. Compare the winning selector's MAE against the best single model MAE.
2. Check whether the same selector also improves MSE.
3. Prefer simpler selectors when improvements are tiny.
4. Treat optional learned stackers as exploratory unless they beat simple validation-selected blends by a clear margin.

The saved metadata includes the winning selector, selected hyperparameters, test MAE/MSE, and the leakage-safe note.